### 실행 순서

이 노트북은 **중간에 커널을 재시작해야** 확인이 되므로 위에서 아래로 한 번에 실행하지 않음

```
1 → 2 → 3 → 4 → 5 → 6                        저장 파트
        ↓
   커널 재시작
        ↓
1(load_dotenv) → 3 → 4 → 5(config 정의만)    사라진 객체 복구
        ↓
7                                            복원 파트
```

- 5번의 `invoke` 셀은 재시작 뒤에 다시 실행하지 않음. 다시 실행하면 파일에서 복원한 것인지 방금 넣은 것인지 구분되지 않음
- 7번은 재시작 뒤에 처음 실행함. 반복해서 돌릴 필요 없음
- 처음부터 다시 테스트하려면 `checkpoints.db`, `checkpoints.db-wal`, `checkpoints.db-shm` 을 지우고 1번부터 시작함

### 1. 패키지 설치 + 환경변수 로드

SQLite 체크포인터는 별도 패키지임

In [1]:
%pip install -qU langchain langchain_openai langgraph langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. InMemorySaver의 한계

`InMemorySaver` 는 RAM에 저장하므로 **커널을 재시작하면 대화가 사라짐**. 실제 서비스에서는 파일이나 DB에 저장하는 체크포인터가 필요함

| 체크포인터 | 저장 위치 | 용도 |
|---|---|---|
| `InMemorySaver` | RAM | 학습, 테스트 |
| `SqliteSaver` | 로컬 파일 | 로컬 개발, 실험 |
| `PostgresSaver` | PostgreSQL | 운영 |

### 3. SqliteSaver 연결

`sqlite3.Connection` 을 넘기면 끝임. 테이블은 첫 저장 시점에 자동으로 생성됨

In [3]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)   # 내부에서 lock으로 직렬화하므로 False로 열어도 됨
memory = SqliteSaver(conn)   # setup()은 직접 호출하지 않음 (필요할 때 내부에서 실행됨)

### 4. 그래프 구성

In [4]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(checkpointer=memory)   # 연결만 바꿨을 뿐 그래프 코드는 동일함

### 5. 대화 진행

In [5]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}   # 재시작 뒤에도 같은 thread_id를 써야 대화가 이어짐

In [6]:
result = graph.invoke({"messages": [("user", "내 이름은 길동이야. 기억해줘")]}, config)
print(result["messages"][-1].content)

안녕, 길동이! 만나서 반가워. 어떤 이야기를 나눌까?


### 6. 파일에 저장됐는지 확인

테이블에 체크포인트가 쌓여 있음

> 체크포인터가 WAL 모드로 열기 때문에 방금 쓴 내용은 `checkpoints.db` 본체가 아니라 `checkpoints.db-wal` 에 먼저 들어감. 본체 크기만 보면 거의 변하지 않으므로 저장 여부는 테이블을 조회해서 확인함

In [7]:
import os

for name in ["checkpoints.db", "checkpoints.db-wal"]:
    if os.path.exists(name):
        print(f"{name}: {os.path.getsize(name)} bytes")

rows = conn.execute("SELECT thread_id, checkpoint_id FROM checkpoints").fetchall()
print(f"\n체크포인트 {len(rows)}개")
for row in rows:
    print(row)

checkpoints.db: 4096 bytes
checkpoints.db-wal: 65952 bytes

체크포인트 3개
('1', '1f191a11-36b1-6bf2-bfff-db197d6d0e7f')
('1', '1f191a11-36b9-60ed-8000-f10a9cd2cbe1')
('1', '1f191a11-46da-628b-8001-dfd2df9540e8')


### 7. 재시작 후 복원 확인

**여기서 커널을 재시작함**

최상단 실행 순서대로 1(`load_dotenv`)·3·4·5(`config` 정의)를 다시 실행한 뒤 아래 셀을 실행하면, 프로세스가 바뀌었는데도 이전 대화를 기억하고 있음

In [6]:
result = graph.invoke({"messages": [("user", "내 이름이 뭐라고 했지?")]}, config)
print(result["messages"][-1].content)

너의 이름은 길동이야.


### 8. 정리

- 체크포인터 **인스턴스만 교체**하면 그래프 코드는 그대로 두고 저장소를 바꿀 수 있음
- `SqliteSaver` 는 `sqlite3.Connection` 만 넘기면 되고, 테이블은 첫 저장 시점에 자동 생성됨 → 프로세스가 죽어도 대화가 남음
- 운영에서는 `PostgresSaver` 등 DB 기반 체크포인터를 씀. 이쪽은 연결 후 `setup()` 을 직접 호출해야 함
- 참고: [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers)